# 📊 RAG Retrieval Evaluation

**Measure and improve your RAG system's retrieval quality**

---

## 📋 Overview

**What you'll learn:**
- Key retrieval metrics (Recall@K, MRR, NDCG)
- Building evaluation datasets
- Automated evaluation pipeline
- A/B testing retrieval strategies
- Production monitoring

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np
from typing import List, Dict, Tuple
import pandas as pd

print("✅ Setup complete")

## 🎯 Why Evaluate Retrieval?

**Bad retrieval = Bad RAG answers**

### The Chain:
```
Query → Retrieval → Context → LLM → Answer
           ↓
    If this is wrong,
    everything fails!
```

### Common Issues:
- 🔍 **Low recall**: Missing relevant docs
- 📊 **Poor ranking**: Relevant docs ranked low
- 🎯 **Wrong chunks**: Retrieved text lacks context
- 💨 **Noise**: Irrelevant results included

### Goal:
Measure → Improve → Measure again

## 📊 Key Metrics

### 1. Recall@K
**"Of all relevant docs, how many did we retrieve in top K?"**
```python
Recall@5 = (Relevant docs in top 5) / (Total relevant docs)
```

### 2. Precision@K
**"Of top K retrieved, how many are relevant?"**
```python
Precision@5 = (Relevant docs in top 5) / 5
```

### 3. MRR (Mean Reciprocal Rank)
**"How high is the first relevant result?"**
```python
MRR = 1 / (rank of first relevant doc)
```

### 4. NDCG (Normalized Discounted Cumulative Gain)
**"Quality of ranking (higher relevance ranked higher)"**

## 🏗️ Building Evaluation Dataset

In [ ]:
# Sample documents
documents = [
    "Python is a high-level programming language for general-purpose programming.",
    "Machine learning is a subset of artificial intelligence.",
    "FastAPI is a modern web framework for building APIs with Python.",
    "NumPy is the fundamental package for scientific computing with Python.",
    "Pandas provides data structures for data analysis in Python.",
    "TensorFlow is an open-source machine learning framework.",
    "React is a JavaScript library for building user interfaces.",
    "Docker is a platform for developing and running containerized applications.",
    "Kubernetes orchestrates containerized applications across clusters.",
    "PostgreSQL is a powerful open-source relational database.",
]

# Create evaluation dataset
# Each test case has: query + relevant document IDs
eval_dataset = [
    {
        "query": "What is Python used for?",
        "relevant_doc_ids": [0, 2, 3, 4]  # Python-related docs
    },
    {
        "query": "Tell me about machine learning frameworks",
        "relevant_doc_ids": [1, 5]  # ML-related docs
    },
    {
        "query": "How to deploy applications?",
        "relevant_doc_ids": [7, 8]  # Container-related docs
    },
    {
        "query": "Python data analysis libraries",
        "relevant_doc_ids": [3, 4]  # NumPy, Pandas
    },
]

print("📊 Evaluation Dataset Created")
print(f"Documents: {len(documents)}")
print(f"Test queries: {len(eval_dataset)}")
print(f"\nSample test case:")
print(f"Query: {eval_dataset[0]['query']}")
print(f"Relevant docs: {eval_dataset[0]['relevant_doc_ids']}")

## 🧮 Implementing Metrics

In [ ]:
class RetrievalMetrics:
    """Calculate retrieval evaluation metrics."""
    
    @staticmethod
    def recall_at_k(retrieved_ids: List[int], relevant_ids: List[int], k: int) -> float:
        """Calculate Recall@K."""
        if not relevant_ids:
            return 0.0
        
        retrieved_set = set(retrieved_ids[:k])
        relevant_set = set(relevant_ids)
        
        hits = len(retrieved_set & relevant_set)
        return hits / len(relevant_set)
    
    @staticmethod
    def precision_at_k(retrieved_ids: List[int], relevant_ids: List[int], k: int) -> float:
        """Calculate Precision@K."""
        if k == 0:
            return 0.0
        
        retrieved_set = set(retrieved_ids[:k])
        relevant_set = set(relevant_ids)
        
        hits = len(retrieved_set & relevant_set)
        return hits / k
    
    @staticmethod
    def mrr(retrieved_ids: List[int], relevant_ids: List[int]) -> float:
        """Calculate Mean Reciprocal Rank."""
        relevant_set = set(relevant_ids)
        
        for rank, doc_id in enumerate(retrieved_ids, start=1):
            if doc_id in relevant_set:
                return 1.0 / rank
        
        return 0.0
    
    @staticmethod
    def average_precision(retrieved_ids: List[int], relevant_ids: List[int]) -> float:
        """Calculate Average Precision."""
        if not relevant_ids:
            return 0.0
        
        relevant_set = set(relevant_ids)
        precisions = []
        num_hits = 0
        
        for rank, doc_id in enumerate(retrieved_ids, start=1):
            if doc_id in relevant_set:
                num_hits += 1
                precision = num_hits / rank
                precisions.append(precision)
        
        if not precisions:
            return 0.0
        
        return sum(precisions) / len(relevant_ids)

# Test metrics
metrics = RetrievalMetrics()

# Example: Retrieved [0, 2, 7, 3, 1], Relevant [0, 2, 3, 4]
retrieved = [0, 2, 7, 3, 1]
relevant = [0, 2, 3, 4]

print("📊 Metric Examples\n")
print(f"Retrieved: {retrieved}")
print(f"Relevant:  {relevant}\n")

print(f"Recall@3:    {metrics.recall_at_k(retrieved, relevant, 3):.3f}")
print(f"Recall@5:    {metrics.recall_at_k(retrieved, relevant, 5):.3f}")
print(f"Precision@3: {metrics.precision_at_k(retrieved, relevant, 3):.3f}")
print(f"Precision@5: {metrics.precision_at_k(retrieved, relevant, 5):.3f}")
print(f"MRR:         {metrics.mrr(retrieved, relevant):.3f}")
print(f"AP:          {metrics.average_precision(retrieved, relevant):.3f}")

## 🔬 Evaluating a RAG System

In [ ]:
class RAGEvaluator:
    """Evaluate RAG retrieval performance."""
    
    def __init__(self, documents: List[str], model_name: str = 'all-MiniLM-L6-v2'):
        self.documents = documents
        self.model = SentenceTransformer(model_name)
        self.embeddings = self.model.encode(documents)
        self.metrics = RetrievalMetrics()
    
    def retrieve(self, query: str, top_k: int = 5) -> List[int]:
        """Retrieve top K document IDs."""
        query_embedding = self.model.encode([query])[0]
        
        # Cosine similarity
        similarities = np.dot(self.embeddings, query_embedding) / (
            np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        
        # Get top k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        return top_indices.tolist()
    
    def evaluate_query(
        self,
        query: str,
        relevant_ids: List[int],
        k_values: List[int] = [1, 3, 5, 10]
    ) -> Dict:
        """Evaluate single query."""
        retrieved_ids = self.retrieve(query, top_k=max(k_values))
        
        results = {
            'query': query,
            'retrieved': retrieved_ids,
            'relevant': relevant_ids,
        }
        
        # Calculate metrics for different K values
        for k in k_values:
            results[f'recall@{k}'] = self.metrics.recall_at_k(retrieved_ids, relevant_ids, k)
            results[f'precision@{k}'] = self.metrics.precision_at_k(retrieved_ids, relevant_ids, k)
        
        results['mrr'] = self.metrics.mrr(retrieved_ids, relevant_ids)
        results['ap'] = self.metrics.average_precision(retrieved_ids, relevant_ids)
        
        return results
    
    def evaluate_dataset(self, eval_dataset: List[Dict]) -> pd.DataFrame:
        """Evaluate full dataset."""
        results = []
        
        for test_case in eval_dataset:
            result = self.evaluate_query(
                test_case['query'],
                test_case['relevant_doc_ids']
            )
            results.append(result)
        
        return pd.DataFrame(results)

# Evaluate
evaluator = RAGEvaluator(documents)

print("🔬 Evaluating RAG System...\n")
results_df = evaluator.evaluate_dataset(eval_dataset)

# Show per-query results
print("Per-Query Results:")
print(results_df[['query', 'recall@3', 'recall@5', 'mrr']].to_string(index=False))

# Show averages
print("\n📊 Average Metrics:")
metric_cols = ['recall@1', 'recall@3', 'recall@5', 'precision@1', 'precision@3', 'mrr', 'ap']
for col in metric_cols:
    if col in results_df.columns:
        print(f"  {col:15} {results_df[col].mean():.3f}")

## 🆚 A/B Testing Retrieval Strategies

In [ ]:
# Compare different embedding models
models_to_compare = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
]

print("🆚 A/B Testing Different Models\n")
print("="*70)

comparison_results = []

for model_name in models_to_compare:
    print(f"\nTesting {model_name}...")
    
    evaluator = RAGEvaluator(documents, model_name=model_name)
    results_df = evaluator.evaluate_dataset(eval_dataset)
    
    # Calculate averages
    avg_metrics = {
        'model': model_name,
        'recall@3': results_df['recall@3'].mean(),
        'recall@5': results_df['recall@5'].mean(),
        'mrr': results_df['mrr'].mean(),
        'map': results_df['ap'].mean(),
    }
    
    comparison_results.append(avg_metrics)
    print(f"  Recall@5: {avg_metrics['recall@5']:.3f}")
    print(f"  MRR: {avg_metrics['mrr']:.3f}")

# Show comparison
print("\n📊 Model Comparison:\n")
comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

best_model = comparison_df.loc[comparison_df['recall@5'].idxmax(), 'model']
print(f"\n🏆 Best model: {best_model}")

## 📈 Production Monitoring

In [ ]:
import json
from datetime import datetime
from pathlib import Path

class RetrievalMonitor:
    """Monitor retrieval quality in production."""
    
    def __init__(self, log_dir: str = "retrieval_logs"):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)
        self.metrics = RetrievalMetrics()
    
    def log_retrieval(
        self,
        query: str,
        retrieved_ids: List[int],
        relevant_ids: List[int] = None,
        user_feedback: str = None
    ):
        """Log a retrieval event."""
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'query': query,
            'retrieved_ids': retrieved_ids,
        }
        
        # If we have ground truth, calculate metrics
        if relevant_ids:
            log_entry['relevant_ids'] = relevant_ids
            log_entry['recall@3'] = self.metrics.recall_at_k(retrieved_ids, relevant_ids, 3)
            log_entry['mrr'] = self.metrics.mrr(retrieved_ids, relevant_ids)
        
        if user_feedback:
            log_entry['user_feedback'] = user_feedback
        
        # Write to log file
        log_file = self.log_dir / f"retrieval_{datetime.now().strftime('%Y%m%d')}.jsonl"
        with open(log_file, 'a') as f:
            f.write(json.dumps(log_entry) + '\n')
    
    def get_daily_stats(self, date: str = None) -> Dict:
        """Get statistics for a day."""
        if date is None:
            date = datetime.now().strftime('%Y%m%d')
        
        log_file = self.log_dir / f"retrieval_{date}.jsonl"
        
        if not log_file.exists():
            return {'error': 'No logs for this date'}
        
        logs = []
        with open(log_file, 'r') as f:
            for line in f:
                logs.append(json.loads(line))
        
        # Calculate stats
        total_queries = len(logs)
        
        # Filter logs with metrics
        logs_with_metrics = [l for l in logs if 'recall@3' in l]
        
        if logs_with_metrics:
            avg_recall = np.mean([l['recall@3'] for l in logs_with_metrics])
            avg_mrr = np.mean([l['mrr'] for l in logs_with_metrics])
        else:
            avg_recall = avg_mrr = None
        
        return {
            'date': date,
            'total_queries': total_queries,
            'queries_with_metrics': len(logs_with_metrics),
            'avg_recall@3': avg_recall,
            'avg_mrr': avg_mrr,
        }

# Test monitoring
monitor = RetrievalMonitor()

# Simulate some retrievals
print("📈 Simulating production retrievals...\n")

for test_case in eval_dataset:
    retrieved = evaluator.retrieve(test_case['query'], top_k=5)
    monitor.log_retrieval(
        test_case['query'],
        retrieved,
        test_case['relevant_doc_ids']
    )

# Get stats
stats = monitor.get_daily_stats()
print("📊 Today's Stats:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

## ✅ Summary

### Key Metrics:

**Recall@K** (Most Important)
```python
# Did we find the relevant docs?
Recall@5 = 0.80  # Found 80% of relevant docs in top 5
```

**Precision@K**
```python
# How many retrieved are actually relevant?
Precision@5 = 0.60  # 3 out of 5 were relevant
```

**MRR**
```python
# How high is first relevant result?
MRR = 1/2 = 0.50  # First relevant at rank 2
```

### Target Metrics:

| Use Case | Recall@5 | Recall@10 | MRR |
|----------|----------|-----------|-----|
| **General Q&A** | > 0.70 | > 0.85 | > 0.50 |
| **Customer Support** | > 0.80 | > 0.90 | > 0.60 |
| **Research** | > 0.90 | > 0.95 | > 0.70 |

### Evaluation Workflow:

```python
# 1. Create eval dataset
eval_set = [
    {"query": "...", "relevant_ids": [...]},
    ...
]

# 2. Run evaluation
results = evaluator.evaluate_dataset(eval_set)

# 3. Analyze
print(f"Recall@5: {results['recall@5'].mean()}")

# 4. Improve
# - Try different embedding models
# - Adjust chunk size
# - Add reranking
# - Use hybrid search

# 5. Re-evaluate
```

### Best Practices:

1. **Build eval set with real queries**
   - Use production logs
   - Cover edge cases
   - Include hard examples

2. **Evaluate regularly**
   - Before deployments
   - After data updates
   - Weekly in production

3. **Track over time**
   - Monitor for degradation
   - A/B test improvements
   - Set up alerts

4. **Focus on Recall@K first**
   - Most important metric
   - If recall is low, LLM can't help
   - Aim for > 0.80

### Improving Low Metrics:

**Low Recall (<0.70):**
- Use better embedding model
- Add hybrid search (keyword + semantic)
- Adjust chunk size
- Add query expansion

**Low MRR (<0.50):**
- Add reranking
- Fine-tune embeddings
- Better chunk overlap

**High variance:**
- More diverse eval set
- Query understanding
- Better preprocessing

### Production Monitoring:

```python
# Log every retrieval
monitor.log_retrieval(
    query=query,
    retrieved_ids=retrieved,
    relevant_ids=ground_truth,  # if available
    user_feedback=feedback       # thumbs up/down
)

# Daily dashboard
stats = monitor.get_daily_stats()
# Alert if recall drops below threshold
```

### Next: `05_rag_systems/06_reranking.ipynb`